In [1]:
from mltau.tools.evaluation import decode_ParTauDETR as dp
from mltau.tools.evaluation import set_to_set_models as sp
import awkward as ak

In [2]:
from mltau.tools.tau_id import TauIDClass, tau_id_classes

In [ ]:
from importlib import reload
reload(dp)
reload(sp)

<module 'mltau.tools.evaluation.set_to_set_models' from '/home/karl/ml-tau-model/mltau/tools/evaluation/set_to_set_models.py'>

In [4]:
import glob
import os.path

run_basename = '20261104_ParTauDETR_4classes'
checkpoint_path = os.path.expanduser(f"~/runs/{run_basename}/models/ParTauDETR-model_best.ckpt")
model = dp.load_model(checkpoint_path)
cfg = model.cfg
class_names, background_samples = tau_id_classes(cfg)
dataset_dir = os.path.expanduser(str(cfg.dataset.data_dir))
signal_data_paths = sorted(glob.glob(f"{dataset_dir}/z_test_*.parquet"))
background_data_paths = {
    sample: sorted(glob.glob(f"{dataset_dir}/{sample}_test_*.parquet"))
    for sample in background_samples
}
if not signal_data_paths:
    raise FileNotFoundError(f"No z test files in {dataset_dir}")
for sample, paths in background_data_paths.items():
    if not paths:
        raise FileNotFoundError(f"No {sample} test files in {dataset_dir}")

In [5]:
import os
import shutil
import numpy as np
import pyarrow.parquet as pq
import torch
from concurrent.futures import ThreadPoolExecutor

num_workers = 10
obj_cls_trsh = 0.5
output_dir = os.path.expanduser(f'~/tmp/{run_basename}-model_best-{obj_cls_trsh:.3f}'.replace('.', 'p') + 'thr')


def merge_parquet_parts(part_paths, output_path):
    temporary_output_path = f'{output_path}.tmp'
    writer = None
    try:
        for part_path in part_paths:
            table = pq.read_table(part_path)
            if writer is None:
                writer = pq.ParquetWriter(temporary_output_path, table.schema)
            writer.write_table(table)
    finally:
        if writer is not None:
            writer.close()

    os.replace(temporary_output_path, output_path)


def infer_part(input_path, sample_name, signal):
    outputs, targets, weights, reco_jet_p4s, data = dp.model_inference(
        checkpoint_path, input_path, cfg, model=model,
    )
    tau_id_probs = dp.tau_id_probabilities(outputs).cpu().numpy()
    tau_tagging_score = np.ascontiguousarray(tau_id_probs[:, TauIDClass.HADRONIC_TAU])

    if signal:
        true_daughters, pred_daughters = dp.create_predictions(
            outputs, targets, weights, reco_jet_p4s, cfg,
            obj_cls_trsh=obj_cls_trsh, tau_threshold=None,
        )
        data_to_save = sp.construct_prediction_file_content(
            data, pred_daughters, true_daughters, cfg.dataset.tau_daughter_pdg_ids,
            tau_tagging_score=tau_tagging_score, debug=True,
        )
    else:
        data_to_save = sp.construct_background_prediction_file_content(
            data, tau_tagging_score=tau_tagging_score,
        )

    for class_index, class_name in enumerate(class_names):
        data_to_save = ak.with_field(
            data_to_save, np.ascontiguousarray(tau_id_probs[:, class_index]),
            f'tau_id_prob_{class_name}',
        )
    parts_dir = os.path.join(output_dir, f'{sample_name}-parts')
    os.makedirs(parts_dir, exist_ok=True)
    part_path = os.path.join(parts_dir, os.path.basename(input_path))
    ak.to_parquet(data_to_save, part_path)
    return part_path


def merge_sample_predictions(sample_name, part_paths):
    output_path = os.path.join(output_dir, f'{sample_name}_test_predictions.parquet')
    os.makedirs(output_dir, exist_ok=True)
    merge_parquet_parts(part_paths, output_path)
    shutil.rmtree(os.path.join(output_dir, f'{sample_name}-parts'))
    return output_path

In [6]:
if num_workers < 1:
    raise ValueError('num_workers must be at least 1')
if num_workers > 1:
    torch.set_num_threads(1)

jobs = [('z', path, True) for path in signal_data_paths]
jobs.extend(
    (sample, path, False)
    for sample, paths in background_data_paths.items()
    for path in paths
)
with ThreadPoolExecutor(max_workers=num_workers) as executor:
    part_paths = list(executor.map(
        lambda job: infer_part(job[1], job[0], job[2]), jobs,
    ))
parts_by_sample = {'z': []}
parts_by_sample.update({sample: [] for sample in background_data_paths})
for (sample, _, _), part_path in zip(jobs, part_paths):
    parts_by_sample[sample].append(part_path)

signal_output_path = merge_sample_predictions('z', parts_by_sample['z'])
background_output_paths = {
    sample: merge_sample_predictions(sample, parts_by_sample[sample])
    for sample in background_data_paths
}

Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 90,661 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
Read 100,000 jets from 1 parquet files.
R

In [7]:
print('Signal predictions:', signal_output_path)
for sample, output_path in background_output_paths.items():
    print(f'{sample} predictions:', output_path)

Signal predictions: /home/karl/tmp/20261104_ParTauDETR_4classes-model_best-0p500thr/z_test_predictions.parquet
qq predictions: /home/karl/tmp/20261104_ParTauDETR_4classes-model_best-0p500thr/qq_test_predictions.parquet
mm predictions: /home/karl/tmp/20261104_ParTauDETR_4classes-model_best-0p500thr/mm_test_predictions.parquet
bhabha predictions: /home/karl/tmp/20261104_ParTauDETR_4classes-model_best-0p500thr/bhabha_test_predictions.parquet
